# 08 · RQ3 and RQ4: Which modality is more useful? Do speakers differ?

Report sections 3.4 (RQ3: Figure 6, Tables 7 and 8) and 3.5 (RQ4: Figure 7,
Table 9). The weighted fusion used in Table 7 and in RQ4 is learned on the
validation set (same method as notebook 09).

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    P = Path("/content/drive/MyDrive/Affective Project/predictions")
else:
    P = Path("../predictions")

FILES = {
    "audio_v2_test": P / "audio_v2_test_predictions.csv",
    "audio_v2_validation": P / "audio_v2_validation_predictions.csv",
    "audio_v1_validation": P / "audio_v1_validation_predictions.csv",
    "audio_v2_history": P / "audio_v2_history.csv",
    "visual_test": P / "visual_test_predictions.csv",
    "visual_validation": P / "visual_validation_predictions.csv",
}
missing = [str(p) for p in FILES.values() if not p.exists()]
assert not missing, "Missing files:\n" + "\n".join(missing)
for k, p in FILES.items():
    print(f"{k:22s} {p}")

Mounted at /content/drive
audio_v2_test          /content/drive/MyDrive/Affective Project/predictions/audio_v2_test_predictions.csv
audio_v2_validation    /content/drive/MyDrive/Affective Project/predictions/audio_v2_validation_predictions.csv
audio_v1_validation    /content/drive/MyDrive/Affective Project/predictions/audio_v1_validation_predictions.csv
audio_v2_history       /content/drive/MyDrive/Affective Project/predictions/audio_v2_history.csv
visual_test            /content/drive/MyDrive/Affective Project/predictions/visual_test_predictions.csv
visual_validation      /content/drive/MyDrive/Affective Project/predictions/visual_validation_predictions.csv


In [2]:
D = Path("/content/drive/MyDrive/Affective Project/data") if IN_COLAB else Path("../data")
DATA = {"ratings": D / "summaryTable.csv", "actors": D / "VideoDemographics.csv"}
missing = [str(p) for p in DATA.values() if not p.exists()]
assert not missing, "Missing files:\n" + "\n".join(missing)

In [3]:
CLASSES = ["Anger", "Disgust", "Fear", "Happy", "Neutral", "Sad"]
PROB = [f"prob_{c}" for c in CLASSES]


def load_split(split):
    """One row per clip: true label and both models' predictions and probabilities.
    Clips without a visual prediction are dropped (only 1076_MTI_SAD_XX in validation)."""
    v = pd.read_csv(FILES[f"visual_{split}"])
    v = v[v["status"] == "ok"]
    a = pd.read_csv(FILES[f"audio_v2_{split}"])
    a["video_id"] = a["filename"].str.replace(".wav", "", regex=False)
    m = v[["video_id", "Speaker", "true_label", "predicted_label", *PROB]].merge(
        a[["video_id", "true_label", "predicted_label", *PROB]],
        on="video_id", suffixes=("_v", "_a"), validate="one_to_one")
    assert (m["true_label_v"] == m["true_label_a"]).all(), "label mismatch"
    m = m.rename(columns={"true_label_v": "y", "predicted_label_v": "pred_v",
                          "predicted_label_a": "pred_a"}).drop(columns="true_label_a")
    return m


def ece(p, y_idx, bins=10):
    """Expected calibration error with 10 equal-width confidence bins."""
    conf, pred = p.max(1), p.argmax(1)
    correct = (pred == y_idx).astype(float)
    edges = np.linspace(0, 1, bins + 1)
    total = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        s = (conf > lo) & (conf <= hi)
        if s.any():
            total += s.mean() * abs(conf[s].mean() - correct[s].mean())
    return total

In [4]:
from scipy.optimize import minimize_scalar
from sklearn.metrics import accuracy_score
EPS = 1e-12


def temp_scale(p, T):
    z = np.log(p + EPS) / T
    z -= z.max(1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(1, keepdims=True)


def fit_temperature(p, y):
    nll = lambda T: -np.mean(np.log(temp_scale(p, T)[np.arange(len(y)), y] + EPS))
    return minimize_scalar(nll, bounds=(0.05, 20), method="bounded").x


def weighted_fusion(val, test):
    """Fit on the validation set, return the fused predictions for the test set."""
    to_idx = lambda s: np.array([CLASSES.index(c) for c in s])
    pv_val, pa_val = val[[f"{p}_v" for p in PROB]].to_numpy(), val[[f"{p}_a" for p in PROB]].to_numpy()
    pv_te, pa_te = test[[f"{p}_v" for p in PROB]].to_numpy(), test[[f"{p}_a" for p in PROB]].to_numpy()
    y_val = to_idx(val.y)
    Tv, Ta = fit_temperature(pv_val, y_val), fit_temperature(pa_val, y_val)
    cv, ca = temp_scale(pv_val, Tv), temp_scale(pa_val, Ta)
    grid = np.linspace(0, 1, 21)
    w = grid[np.argmax([accuracy_score(y_val, (g * cv + (1 - g) * ca).argmax(1)) for g in grid])]
    fused = w * temp_scale(pv_te, Tv) + (1 - w) * temp_scale(pa_te, Ta)
    print(f"Fitted on validation: T_visual = {Tv:.2f}, T_audio = {Ta:.2f}, w_visual = {w:.2f}")
    return np.array(CLASSES)[fused.argmax(1)]

In [5]:
from scipy.stats import binomtest, chi2_contingency, mannwhitneyu, spearmanr, wilcoxon
from sklearn.metrics import f1_score, precision_score

val, m = load_split("validation"), load_split("test")
m["pf"] = weighted_fusion(val, m)
m["cv"], m["ca"], m["cf"] = m.pred_v == m.y, m.pred_a == m.y, m.pf == m.y
m["dis"] = m.pred_v != m.pred_a
m["intensity"] = m.video_id.str.split("_").str[3]
actors = pd.read_csv(DATA["actors"])
m = m.merge(actors, left_on="Speaker", right_on="ActorID", validate="many_to_one")
print("Test clips:", len(m), "| weighted fusion accuracy:", f"{100 * m.cf.mean():.1f}%")

Fitted on validation: T_visual = 2.14, T_audio = 1.12, w_visual = 0.60
Test clips: 1552 | weighted fusion accuracy: 80.0%


# RQ3: Which modality is more useful for each emotion?

## F1 difference (visual − audio) with 95% confidence intervals over speakers (Figure 6a)

The confidence interval resamples whole speakers 2,000 times (speaker bootstrap).

In [6]:
rng = np.random.default_rng(0)
speakers = m.Speaker.unique()
idx_by = {s: np.where(m.Speaker.values == s)[0] for s in speakers}
diffs = []
for _ in range(2000):
    g = m.iloc[np.concatenate([idx_by[s] for s in rng.choice(speakers, len(speakers))])]
    diffs.append(f1_score(g.y, g.pred_v, labels=CLASSES, average=None)
                 - f1_score(g.y, g.pred_a, labels=CLASSES, average=None))
ci = np.percentile(diffs, [2.5, 97.5], axis=0)

f1_v = f1_score(m.y, m.pred_v, labels=CLASSES, average=None)
f1_a = f1_score(m.y, m.pred_a, labels=CLASSES, average=None)
fig6a = pd.DataFrame({"F1_visual": 100 * f1_v, "F1_audio": 100 * f1_a,
                      "difference": 100 * (f1_v - f1_a),
                      "CI_low": 100 * ci[0], "CI_high": 100 * ci[1]}, index=CLASSES).round(1)
fig6a["CI_excludes_0"] = (fig6a.CI_low > 0) | (fig6a.CI_high < 0)
fig6a

,F1_visual,F1_audio,difference,CI_low,CI_high,CI_excludes_0
Anger,66.8,75.1,-8.3,-17.7,1.5,False
Disgust,79.4,61.2,18.2,8.4,28.7,True
Fear,61.6,64.7,-3.1,-11.9,5.6,False
Happy,95.7,68.5,27.2,22.2,32.6,True
Neutral,58.5,74.6,-16.1,-27.2,-5.1,True
Sad,42.0,61.9,-19.9,-33.9,-6.3,True


## McNemar test per emotion (Table 7, column p)

On the clips of each emotion: how many were recognised only by the visual model
and only by the audio model. Holm correction for the six tests.

In [7]:
rows = []
for c in CLASSES:
    g = m[m.y == c]
    only_v, only_a = int((g.cv & ~g.ca).sum()), int((~g.cv & g.ca).sum())
    rows.append({"emotion": c, "recall_visual": 100 * g.cv.mean(), "recall_audio": 100 * g.ca.mean(),
                 "precision_visual": 100 * precision_score(m.y, m.pred_v, labels=[c], average=None)[0],
                 "precision_audio": 100 * precision_score(m.y, m.pred_a, labels=[c], average=None)[0],
                 "only_visual_right": only_v, "only_audio_right": only_a,
                 "p": binomtest(only_v, only_v + only_a, 0.5).pvalue})
mc = pd.DataFrame(rows).set_index("emotion")
order, running, holm = np.argsort(mc.p.values), 0, np.empty(6)
for r, i in enumerate(order):
    running = max(running, min(1, (6 - r) * mc.p.values[i])); holm[i] = running
mc["p_holm"] = holm
mc.round(3)

,recall_visual,recall_audio,precision_visual,precision_audio,only_visual_right,only_audio_right,p,p_holm
emotion,,,,,,,,
Anger,63.396,76.226,70.588,73.993,38,72,0.002,0.003
Disgust,82.264,55.094,76.761,68.868,96,24,0.000,0.000
Fear,66.038,59.245,57.756,71.364,64,46,0.105,0.105
Happy,95.849,65.660,95.489,71.605,86,6,0.000,0.000
Neutral,63.877,87.225,53.903,65.132,18,71,0.000,0.000
Sad,36.226,66.038,50.000,58.333,37,116,0.000,0.000


## Who is right when the models disagree? (Figure 6b, % of the disagreement clips of each emotion)

In [8]:
d = m[m.dis]
who = pd.DataFrame({
    "n_disagreements": d.groupby("y").size(),
    "only_visual_right_%": 100 * (d.cv & ~d.ca).groupby(d.y).mean(),
    "only_audio_right_%": 100 * (~d.cv & d.ca).groupby(d.y).mean(),
    "neither_right_%": 100 * (~d.cv & ~d.ca).groupby(d.y).mean(),
}).loc[CLASSES].round(0)
who

,n_disagreements,only_visual_right_%,only_audio_right_%,neither_right_%
y,,,,
Anger,129,29.0,56.0,15.0
Disgust,135,71.0,18.0,11.0
Fear,140,46.0,33.0,21.0
Happy,96,90.0,6.0,4.0
Neutral,97,19.0,73.0,8.0
Sad,198,19.0,59.0,23.0


## Effect of combining the two models (Table 7, columns Best, Fusion, Change)

In [9]:
f1_f = f1_score(m.y, m.pf, labels=CLASSES, average=None)
table7 = pd.DataFrame({"best_single_F1": 100 * np.maximum(f1_v, f1_a), "fusion_F1": 100 * f1_f},
                      index=CLASSES)
table7["change"] = table7.fusion_F1 - table7.best_single_F1
table7.round(1).loc[["Happy", "Disgust", "Fear", "Anger", "Neutral", "Sad"]]

,best_single_F1,fusion_F1,change
Happy,95.7,94.7,-1.0
Disgust,79.4,81.6,2.2
Fear,64.7,75.8,11.1
Anger,75.1,85.8,10.7
Neutral,74.6,78.2,3.6
Sad,61.9,62.9,1.0


## Effect of emotion intensity (Table 8)

In [10]:
ieo = m[m.intensity.isin(["LO", "MD", "HI"])]
(100 * ieo.groupby("intensity")[["ca", "cv"]].mean()).loc[["LO", "MD", "HI"]].round(1) \
    .rename(columns={"ca": "audio_accuracy_%", "cv": "visual_accuracy_%"}) \
    .assign(clips=ieo.groupby("intensity").size())

,audio_accuracy_%,visual_accuracy_%,clips
intensity,,,
LO,50.5,55.8,95
MD,65.3,64.2,95
HI,68.4,78.9,95


# RQ4: Do different speakers show emotions in different ways?

## Accuracy per speaker (Figure 7a)

Per-speaker McNemar test: is the difference between the two models significant
for this speaker?

In [11]:
sp = m.groupby("Speaker").agg(sex=("Sex", "first"), age=("Age", "first"), race=("Race", "first"),
                              clips=("y", "size"), acc_visual=("cv", "mean"), acc_audio=("ca", "mean"),
                              acc_fusion=("cf", "mean"), disagreement=("dis", "mean"))
sp["diff_v_minus_a"] = sp.acc_visual - sp.acc_audio
pvals = []
for s, g in m.groupby("Speaker"):
    only_v, only_a = int((g.cv & ~g.ca).sum()), int((~g.cv & g.ca).sum())
    pvals.append(binomtest(only_v, only_v + only_a, 0.5).pvalue if only_v + only_a else 1.0)
sp["p_mcnemar"] = pvals
sp["better"] = np.select([sp.diff_v_minus_a > 0, sp.diff_v_minus_a < 0], ["face", "voice"], "equal")
sp["fusion_minus_best"] = sp.acc_fusion - sp[["acc_visual", "acc_audio"]].max(axis=1)
pct = ["acc_visual", "acc_audio", "acc_fusion", "disagreement", "diff_v_minus_a", "fusion_minus_best"]
(sp.assign(**{c: 100 * sp[c] for c in pct}).sort_values("diff_v_minus_a").round(3))

,sex,age,race,clips,acc_visual,acc_audio,acc_fusion,disagreement,diff_v_minus_a,p_mcnemar,better,fusion_minus_best
Speaker,,,,,,,,,,,,
1037,Female,45,Caucasian,82,54.878,75.610,82.927,56.098,-20.732,0.014,voice,7.317
1010,Female,27,Caucasian,82,58.537,74.390,75.610,54.878,-15.854,0.047,voice,1.220
1046,Female,22,Caucasian,82,50.000,63.415,67.073,54.878,-13.415,0.090,voice,3.659
1032,Male,30,African American,82,46.341,57.317,64.634,68.293,-10.976,0.211,voice,7.317
1014,Male,24,Caucasian,82,59.756,68.293,78.049,57.317,-8.537,0.311,voice,9.756
1018,Female,25,Caucasian,82,71.951,79.268,90.244,46.341,-7.317,0.405,voice,10.976
1022,Male,22,Caucasian,82,68.293,75.610,81.707,43.902,-7.317,0.362,voice,6.098
1017,Male,42,Caucasian,82,53.659,60.976,59.756,62.195,-7.317,0.405,voice,-1.220
1005,Male,29,African American,82,57.317,62.195,69.512,62.195,-4.878,0.618,voice,7.317


## 3.5.1 Speakers differ in how easy they are to recognise

In [12]:
for name, col, sc in [("Visual accuracy", "cv", "acc_visual"), ("Audio accuracy", "ca", "acc_audio"),
                      ("Disagreement", "dis", "disagreement")]:
    chi2, p, dof, _ = chi2_contingency(pd.crosstab(m.Speaker, m[col]))
    print(f"{name:16s}: range {100 * sp[sc].min():.1f}-{100 * sp[sc].max():.1f}%, "
          f"SD {100 * sp[sc].std():.1f} points, speaker effect chi2 = {chi2:.1f}, p = {p:.4f}")

Visual accuracy : range 46.3-87.8%, SD 12.4 points, speaker effect chi2 = 104.8, p = 0.0000
Audio accuracy  : range 56.1-79.3%, SD 6.7 points, speaker effect chi2 = 30.3, p = 0.0346
Disagreement    : range 37.8-68.3%, SD 7.9 points, speaker effect chi2 = 37.1, p = 0.0051


## 3.5.2 Some speakers express emotions more clearly in the face, others in the voice

In [13]:
rho, p = spearmanr(sp.acc_visual, sp.acc_audio)
print(f"Visual vs. audio accuracy over speakers: Spearman rho = {rho:.2f}, p = {p:.2f}")
for s in [1025, 1037]:
    print(f"Speaker {s}: visual {100 * sp.loc[s, 'acc_visual']:.1f}%, audio {100 * sp.loc[s, 'acc_audio']:.1f}%")
print("\nBetter modality per speaker:", sp.better.value_counts().to_dict(),
      "| equal:", list(sp.index[sp.better == "equal"]))
print(f"Wilcoxon test (visual vs. audio over speakers): p = {wilcoxon(sp.acc_visual, sp.acc_audio).pvalue:.2f}")
sig = sp[sp.p_mcnemar < 0.05]
print(f"Significant per speaker (McNemar p < 0.05): {len(sig)} speakers, "
      f"face {list(sig.index[sig.better == 'face'])}, voice {list(sig.index[sig.better == 'voice'])}")

Visual vs. audio accuracy over speakers: Spearman rho = 0.06, p = 0.81
Speaker 1025: visual 86.6%, audio 63.4%
Speaker 1037: visual 54.9%, audio 75.6%

Better modality per speaker: {'voice': 12, 'face': 6, 'equal': 1} | equal: [1079]
Wilcoxon test (visual vs. audio over speakers): p = 0.97
Significant per speaker (McNemar p < 0.05): 6 speakers, face [1025, 1045, 1048, 1075], voice [1010, 1037]


## 3.5.3 Which patterns are shared by all speakers? (Figure 7b)

In [14]:
rec = m.groupby(["Speaker", "y"])[["cv", "ca"]].mean()
diff = (rec.cv - rec.ca).unstack()[CLASSES]
pd.DataFrame({"face_better": (diff > 0).sum(), "voice_better": (diff < 0).sum(),
              "equal": (diff == 0).sum()}).T

y,Anger,Disgust,Fear,Happy,Neutral,Sad
face_better,6,16,13,17,5,5
voice_better,12,3,6,1,12,14
equal,1,0,0,1,2,0


In [15]:
print(f"Anger, speaker 1019: visual - audio recall = {100 * diff.loc[1019, 'Anger']:.0f} points")
print(f"Anger, speaker 1025: visual - audio recall = {100 * diff.loc[1025, 'Anger']:.0f} points")
print(f"\nWeighted fusion better than the speaker's better single model: "
      f"{(sp.fusion_minus_best > 0).sum()} of {len(sp)} speakers")
print("Other speakers (fusion minus better model, points):",
      (100 * sp.fusion_minus_best[sp.fusion_minus_best <= 0]).round(1).to_dict())

Anger, speaker 1019: visual - audio recall = -85 points
Anger, speaker 1025: visual - audio recall = 50 points

Weighted fusion better than the speaker's better single model: 16 of 19 speakers
Other speakers (fusion minus better model, points): {1017: -1.2, 1039: 0.0, 1075: -1.2}


## 3.5.4 Gender and age (Table 9)

In [16]:
rows = []
for name, col in [("Visual accuracy", "acc_visual"), ("Audio accuracy", "acc_audio"),
                  ("Fusion accuracy", "acc_fusion"), ("Disagreement", "disagreement")]:
    f, ml = sp[sp.sex == "Female"][col], sp[sp.sex == "Male"][col]
    rows.append({"": name, f"female ({len(f)})": 100 * f.mean(), f"male ({len(ml)})": 100 * ml.mean(),
                 "p (Mann-Whitney)": mannwhitneyu(f, ml).pvalue})
pd.DataFrame(rows).round(3)

,,female (9),male (10),p (Mann-Whitney)
0,Visual accuracy,69.241,66.945,0.713
1,Audio accuracy,71.003,64.881,0.053
2,Fusion accuracy,84.146,76.290,0.037
3,Disagreement,49.051,53.151,0.437


In [17]:
r = m.groupby(["Speaker", "Sex", "y"])[["cv", "ca"]].mean().reset_index()
tests = []
for c in CLASSES:
    for mod, col in [("visual", "cv"), ("audio", "ca")]:
        g = r[r.y == c]
        tests.append((c, mod, mannwhitneyu(g[g.Sex == "Female"][col], g[g.Sex == "Male"][col]).pvalue))
tests = pd.DataFrame(tests, columns=["emotion", "model", "p"])
print(tests[tests.p < 0.05].round(3).to_string(index=False))
print(f"\n{(tests.p < 0.05).sum()} of 12 tests with p < 0.05; "
      f"with Bonferroni correction (p < 0.05/12): {(tests.p < 0.05 / 12).sum()}")

emotion model     p
   Fear audio 0.048
    Sad audio 0.032

2 of 12 tests with p < 0.05; with Bonferroni correction (p < 0.05/12): 0


In [18]:
for name, col in [("visual accuracy", "acc_visual"), ("audio accuracy", "acc_audio"), ("disagreement", "disagreement")]:
    rho, p = spearmanr(sp.age, sp[col])
    print(f"Age vs. {name}: rho = {rho:.2f}, p = {p:.2f}")
print("\nTest speakers by group:", sp.race.value_counts().to_dict())
n_se = m.groupby(["Speaker", "y"]).size()
print(f"Clips per speaker and emotion (Section 4.5): mean {n_se.mean():.1f}, range {n_se.min()}-{n_se.max()}")

Age vs. visual accuracy: rho = 0.26, p = 0.27
Age vs. audio accuracy: rho = -0.14, p = 0.56
Age vs. disagreement: rho = -0.18, p = 0.46

Test speakers by group: {'Caucasian': 14, 'African American': 3, 'Asian': 2}
Clips per speaker and emotion (Section 4.5): mean 13.6, range 11-14
